# 2026-08-31 세션 실험 — 검토용 노트북

`실험보고서_구조불균형분할_손은총_20260831.docx` 의 모든 표를 **저장된 산출물에서 다시 계산**한다.
모델을 재적합하지 않으므로 몇 초면 끝난다.

무거운 계산은 아래 .py 가 이미 돌려 산출물로 남겼다:
`compare_serial_parallel.py` · `train_binary_heads.py` · `imbalance_search.py` ·
`ablate_classes_blocks.py` · `boundary_and_purging.py` · `pattern_struct_by_set.py`

In [1]:
import sys, json, numpy as np, pandas as pd
sys.path.insert(0, '/workspace')
pd.set_option('display.width', 180); pd.set_option('display.max_columns', 40)
W='/workspace/'
C=lambda p: pd.read_csv(W+p); J=lambda p: json.load(open(W+p))

## 1. 기준 항목 — 직렬 vs 병렬 2×2 (§2)

8칸 중 `L_all × 병렬(순위융합)` 한 칸만 이긴다. 나머지는 α=1.0 으로 수렴해 동일하다.

In [2]:
a = C('model_arch/serial_vs_parallel_test.csv')
a['f1'] = 2*a.precision*a.recall/(a.precision+a.recall)
print('test F1'); display(a.pivot_table(index=['head','arch'],columns='K',values='f1').round(4))
print('val 에서 고른 융합 가중 alpha'); display(a.pivot_table(index=['head','arch'],columns='K',values='kp_frac'))

test F1


K                   300     500     843     1200    2000    3000
head  arch                                                      
L_all 병렬(순위융합)    0.3382  0.4577  0.5065  0.4848  0.4066  0.3268
      병렬(합집합)     0.3313  0.4504  0.4894  0.4789  0.3990  0.3263
      직렬          0.3313  0.4504  0.4894  0.4789  0.3990  0.3263
      직렬(오라클라우팅)  0.3313  0.4504  0.4894  0.4789  0.3990  0.3263
L_oop 병렬(순위융합)    0.3313  0.4504  0.4894  0.4789  0.3990  0.3263
      병렬(합집합)     0.3313  0.4504  0.4894  0.4789  0.3990  0.3263
      직렬          0.3313  0.4504  0.4894  0.4789  0.3990  0.3263
      직렬(오라클라우팅)  0.3313  0.4504  0.4894  0.4789  0.3990  0.3263

val 에서 고른 융합 가중 alpha


K                 300   500   843   1200  2000  3000
head  arch                                          
L_all 병렬(순위융합)     0.4   0.7   0.5   0.8   0.6   0.2
      병렬(합집합)      1.0   1.0   1.0   1.0   1.0   1.0
      직렬           1.0   1.0   1.0   1.0   1.0   1.0
      직렬(오라클라우팅)   1.0   1.0   1.0   1.0   1.0   1.0
L_oop 병렬(순위융합)     1.0   1.0   1.0   1.0   1.0   1.0
      병렬(합집합)      1.0   1.0   1.0   1.0   1.0   1.0
      직렬           1.0   1.0   1.0   1.0   1.0   1.0
      직렬(오라클라우팅)   1.0   1.0   1.0   1.0   1.0   1.0

### 1.1 왜 이겼나 — 건별 인과

융합이 교체한 알림이 무엇이었는지 직접 센다.

In [3]:
from train9 import data as D
b = D.load_basis('10day'); ispos = b.is_pos('te').astype(bool); y9 = b.y('te')
sp = np.load(W+'model_9class/proba_test.npy')[:, :8].max(1)
sl = np.load(W+'model_binary/p_L_all_test.npy')
n, K, alpha = len(sp), 843, 0.5
rp = np.empty(n); rp[np.argsort(-sp, kind='stable')] = np.arange(n)
rl = np.empty(n); rl[np.argsort(-sl, kind='stable')] = np.arange(n)
base = set(np.argsort(-sp, kind='stable')[:K].tolist())
fus  = set(np.argsort(alpha*rp + (1-alpha)*rl, kind='stable')[:K].tolist())
add, rem = np.array(sorted(fus-base)), np.array(sorted(base-fus))
display(pd.DataFrame([
 {'구분':'새로 들어옴','건수':len(add),'세탁':int(ispos[add].sum()),'정밀도':round(float(ispos[add].mean()),3)},
 {'구분':'빠져나감','건수':len(rem),'세탁':int(ispos[rem].sum()),'정밀도':round(float(ispos[rem].mean()),3)},
 {'구분':'순이익','건수':0,'세탁':int(ispos[add].sum()-ispos[rem].sum()),'정밀도':np.nan}]).set_index('구분'))
print(f'새로 들어온 건의 원래 패턴점수 순위: 중앙 {int(np.median(rp[add])):,} · 최소 {int(rp[add].min()):,} '
      f'· 최대 {int(rp[add].max()):,}  (알림 경계 = {K-1:,})')
print(f'두 점수 Spearman 상관 = {pd.Series(rp).corr(pd.Series(rl), method="spearman"):.4f}')

,건수,세탁,정밀도
구분,,,
새로 들어옴,86,29,0.337
빠져나감,86,12,0.140
순이익,0,17,NaN


새로 들어온 건의 원래 패턴점수 순위: 중앙 950 · 최소 844 · 최대 1,838  (알림 경계 = 842)


두 점수 Spearman 상관 = 0.8500


### 1.2 왜 이득이 작나 — 점수 단독 성능

`L_oop`(현재 2차)는 사실상 무력하고, 패턴 외 세탁은 어느 모델로도 안 잡힌다.

In [4]:
from sklearn.metrics import average_precision_score
ipv, y9v = b.is_pos('va').astype(bool), b.y('va')
spv = np.load(W+'model_9class/proba_val.npy')[:, :8].max(1)
pat, oop = (y9v < 8), (ipv & (y9v == 8))
rows=[]
for nm, s in (('1차 패턴점수 s_p', spv),
              ('L_all (전 행 학습)', np.load(W+'model_binary/p_L_all_val.npy')),
              ('L_oop (현재 2차)',  np.load(W+'model_binary/p_L_oop_val.npy'))):
    o=np.argsort(-s); tp=np.cumsum(ipv[o]); k=np.arange(1,len(s)+1)
    P,R = tp/k, tp/ipv.sum(); F = 2*P*R/np.maximum(P+R,1e-12); top=o[:843]
    rows.append({'점수':nm,'AP':round(average_precision_score(ipv,s),4),'최대F1':round(float(F.max()),4),
                 '상위843 패턴세탁':int(pat[top].sum()),'상위843 패턴외세탁':int(oop[top].sum())})
print(f'val 세탁 {int(ipv.sum())} = 패턴 {int(pat.sum())} + 패턴외 {int(oop.sum())}')
display(pd.DataFrame(rows).set_index('점수'))

val 세탁 1083 = 패턴 694 + 패턴외 389


,AP,최대F1,상위843 패턴세탁,상위843 패턴외세탁
점수,,,,
1차 패턴점수 s_p,0.4465,0.5229,495,7
L_all (전 행 학습),0.4171,0.4868,456,7
L_oop (현재 2차),0.0212,0.0667,21,12


## 2. 클래스 불균형 (§3)

20칸 중 원본 전량 + balanced 가 최고. 언더샘플링은 전부 그 아래.

In [5]:
imb = C('model_9class/tables/imbalance_search.csv')
print('사전확률 보정 후 Pl@R70'); display(imb.pivot_table(index=['정상:세탁','method'],columns='w',values='corr_Pl@R70').round(4))
print('보정 전(raw) — 언더샘플할수록 격차가 벌어진다'); display(imb.pivot_table(index=['정상:세탁','method'],columns='w',values='raw_Pl@R70').round(4))
best = imb.loc[imb.groupby(['정상:세탁','method'])['corr_Pl@R70'].mean().idxmax()] if False else None
print('셀 평균 최고:', imb.groupby(['정상:세탁','method'])['corr_Pl@R70'].mean().idxmax(),
      round(imb.groupby(['정상:세탁','method'])['corr_Pl@R70'].mean().max(),4))

사전확률 보정 후 Pl@R70


w                      30       6       9  balanced
정상:세탁     method                                   
100:1     cluster  0.5376  0.5716  0.5733    0.5779
          random   0.5330  0.4958  0.4981    0.5365
10:1      cluster  0.5693  0.5774  0.5762    0.5745
          random   0.5395  0.5739  0.5364    0.5382
300:1     cluster  0.4995  0.4986  0.4953    0.5028
          random   0.5727  0.4991  0.4981    0.5005
30:1      cluster  0.5687  0.5751  0.5687    0.5693
          random   0.5756  0.5335  0.5337    0.5756
원본 1326:1 none     0.5704  0.5774  0.4967    0.5820

보정 전(raw) — 언더샘플할수록 격차가 벌어진다


w                      30       6       9  balanced
정상:세탁     method                                   
100:1     cluster  0.4420  0.4378  0.4084    0.4700
          random   0.4761  0.4761  0.4747    0.4803
10:1      cluster  0.3212  0.3191  0.3203    0.3209
          random   0.3564  0.3577  0.3332    0.3570
300:1     cluster  0.4892  0.4883  0.4873    0.4892
          random   0.4977  0.4939  0.4902    0.4925
30:1      cluster  0.3624  0.3642  0.3638    0.3625
          random   0.4043  0.3997  0.4008    0.4054
원본 1326:1 none     0.5739  0.5739  0.5023    0.5751

셀 평균 최고: ('10:1', 'cluster') 0.5743


## 3. 꼬리(10일 이후) (§4)

꼬리는 test 의 0.109% 인데 세탁의 36.4% 를 담고 채점을 14.3점 부풀린다.

In [6]:
display(pd.DataFrame(J('model_9class/tables/lit_compare_18day.json'))[['name','n','n_pos','max_f1','p','r']].round(4))

,name,n,n_pos,max_f1,p,r
0,18day test 전체(=논문 split),1015788,1798,0.6411,0.7003,0.5912
1,주 기간만,1014680,1143,0.4981,0.5453,0.4584
2,꼬리만,1108,655,0.9430,0.9152,0.9725


## 4. 분할 경계 — purging 과 걸침/단독 (§5·§6)

In [7]:
print('purging 효과'); display(C('model_9class/tables/purging_effect.csv'))
print('걸침 vs 단독'); display(C('model_9class/tables/boundary_split_performance.csv'))

purging 효과


,설정,n_train,패턴행,K@R70,Pa@R70,Pl@R70,fit_s
0,purging 없음(현재),3045987,1176,866,0.5762,0.5855,475
1,purging 적용,3045394,583,1067,0.4733,0.4789,544


걸침 vs 단독


,그룹,시도,test내 평균거래,시도 적발률,거래 단위 재현율
0,걸침(경계 넘음),59,3.2,0.8305,0.6968
1,단독(한 split 안),24,1.7,0.7083,0.6000
2,미완결,100,4.6,0.9000,0.7149
3,== 전체 ==,183,3.7,0.8525,0.7032


### 4.1 코드 버그 — `final_9class.py:107` 이 `eligible` 열을 쓰지 않는다

`eligible` 의 '2건 이상' 조건이 빠져 거래 1건짜리 시도가 채점에 들어간다.

In [8]:
ga = C('processed_9class/HI-Small_10day/blocks/attempt_eligibility.csv')
buggy = set(ga.loc[ga['contained'] & ga['complete'] & (ga['s_min']==2),'att'])
right = set(ga.loc[ga['eligible'].astype(bool) & (ga['s_min']==2),'att'])
idx = b.index_full(); att = idx['attempt'][idx['split']==2]
op = J('model_9class/summary.json')['operating_point_from_val']
al = sp >= op['tau']
df = pd.DataFrame({'att':att,'a':al}); df = df[df['att']>=0]
g, sz = df.groupby('att')['a'].any(), df.groupby('att').size()
out=[]
for nm,S in (('현재 코드 (contained & complete)',buggy), ('올바른 eligible',right)):
    ids=[x for x in S if x in g.index]; d,z = g.loc[ids], sz.loc[ids]
    out.append({'채점 조건':nm,'대상 시도':len(ids),'적발률':round(float(d.mean()),4),
                '거래1건 시도':int((z==1).sum()),'2건이상 시도':int((z>=2).sum())})
display(pd.DataFrame(out).set_index('채점 조건'))

,대상 시도,적발률,거래1건 시도,2건이상 시도
채점 조건,,,,
현재 코드 (contained & complete),24,0.7083,15,9
올바른 eligible,9,1.0000,0,9


## 5. 분할 담김 비율 — 규모 불변 (§5.6)

In [9]:
sc = C('model_blocks/split_containment_by_set.csv')
display(sc.pivot_table(index='클래스',columns='세트',values='담김비율').round(3))
print('평가 구간 길이 vs 채점 가능 시도 (HI-Large)')
display(C('model_blocks/eval_window_curve.csv').query("세트=='HI-Large'").set_index('평가구간(일)'))

세트,HI-Large,HI-Medium,HI-Small
클래스,,,
BIPARTITE,0.618,0.580,0.564
CYCLE,0.251,0.241,0.241
FAN-IN,0.329,0.334,0.270
FAN-OUT,0.345,0.382,0.359
GATHER-SCATTER,0.248,0.242,0.233
RANDOM,0.431,0.409,0.456
SCATTER-GATHER,0.196,0.196,0.238
STACK,0.242,0.250,0.235


평가 구간 길이 vs 채점 가능 시도 (HI-Large)


,세트,겹침,통째담김,담김비율,클래스당 담김(중앙)
평가구간(일),,,,,
7,HI-Large,1216,210,0.173,31
14,HI-Large,2417,505,0.209,67
21,HI-Large,3574,934,0.261,105
28,HI-Large,4791,1480,0.309,160
35,HI-Large,5930,2277,0.384,269
49,HI-Large,8218,4485,0.546,571
70,HI-Large,10989,7813,0.711,983


## 6. BIPARTITE 구조와 클래스 제외 실험 (§7)

계좌 공유 조각 수는 세 세트 모두 같다 — 규모로 해결되지 않는다.

In [10]:
st = C('model_blocks/pattern_structure_by_set.csv')
display(st.pivot(index='클래스',columns='세트',values='평균조각(계좌공유)').round(2))
print('시도 span (팀원 보고 검증)')
display(C('model_blocks/attempt_span_by_set.csv').query("클래스=='== 전체 =='").set_index('세트'))
print('클래스별 F1 — 오라클 블록이면 BIPARTITE 0.928, 운영이면 0.000')
display(C('model_blocks/per_class_by_testdef.csv').rename(columns={'Unnamed: 0':'클래스'}).set_index('클래스').round(3))
print('클래스 제외 ablation — 같은 라벨 집합으로 채점하면 이득 0')
display(C('model_blocks/ablate_classes_results.csv')
        .pivot(index='설정',columns='test_def',values='edge_cond_macro_f1')[['attempt','window','pred']].round(4))

세트,HI-Large,HI-Medium,HI-Small
클래스,,,
BIPARTITE,5.77,5.79,5.37
CYCLE,1.00,1.00,1.00
FAN-IN,1.00,1.00,1.00
FAN-OUT,1.00,1.00,1.00
GATHER-SCATTER,1.00,1.00,1.00
RANDOM,1.00,1.00,1.00
SCATTER-GATHER,1.00,1.00,1.00
STACK,5.51,5.84,5.42


시도 span (팀원 보고 검증)


,클래스,시도,span중앙,span평균,span P90,span최대,span>30일
세트,,,,,,,
HI-Small,== 전체 ==,370,3.1,2.8,4.6,8.4,0.000
HI-Medium,== 전체 ==,2756,4.6,4.1,6.0,12.7,0.000
HI-Large,== 전체 ==,16467,27.3,24.1,37.1,70.5,0.423


클래스별 F1 — 오라클 블록이면 BIPARTITE 0.928, 운영이면 0.000


,attempt F1,attempt n,window F1,window n,pred F1,pred n
클래스,,,,,,
FAN-OUT,0.819,74,0.634,74,0.595,58
FAN-IN,0.667,71,0.663,71,0.692,51
CYCLE,0.547,65,0.596,65,0.208,43
SCATTER-GATHER,0.764,123,0.712,123,0.690,91
GATHER-SCATTER,0.627,147,0.481,147,0.519,109
BIPARTITE,0.928,52,0.000,52,0.000,30
STACK,0.940,107,0.808,107,0.661,72
RANDOM,0.575,45,0.405,45,0.252,22


클래스 제외 ablation — 같은 라벨 집합으로 채점하면 이득 0


test_def,attempt,window,pred
설정,,,
-BIPARTITE,0.7014,0.6245,0.5291
-BIPARTITE (기준: 8종 학습 모델을 같은 라벨로 채점),0.7095,0.6369,0.5291
-BIPARTITE-STACK,0.6780,0.6251,0.5204
-BIPARTITE-STACK (기준: 8종 학습 모델을 같은 라벨로 채점),0.6765,0.6274,0.5305
-STACK,0.7098,0.4782,0.4304
-STACK (기준: 8종 학습 모델을 같은 라벨로 채점),0.7082,0.5091,0.4348
전체 8종,0.7332,0.5374,0.4524


## 7. 읽을 때 붙는 단서

1. **test 재사용** — 이 세션은 test 를 여러 각도로 열었다. 운영점·설정 선택은 전부 val 로 했으나,
   앞으로 이 test 로 새 설정을 고르면 그 수치는 못 쓴다.
2. **불균형 탐색의 seed 수가 다르다** — 원본(full) 칸은 seed 1회, 언더샘플 칸은 2회다(비용).
   08-27 `corrected_seedvar.csv` 의 seed 5회 결과와 방향이 일치하는지 함께 봐야 한다.
3. **사건 단위 지표는 표본이 없다** — test 에 채점 자격 시도가 9건(2.5%)뿐이다. 성능 근거로 쓸 수 없다.
4. **HI-Large 는 거래 파일이 0.6% 만 있다** — 패턴 구조·span·분할 시뮬레이션만 유효하고 학습은 불가.